# SA-CUT — Colab training (Ch5 main experiment)

Trains **SA-CUT** for the Ch5 experiment matrix. E1 (the full model, reference run that
baselines E2 and ablations E3 compare against) finished at epoch 399; the architecture
currently under test is **v2**, which replaces the checkerboard-prone transposed-conv
decoder with resize-conv.

**Which experiment runs is set once, in Cell 1** (`CONFIG` / `SHORT_NAME` / `RUN_NAME`).
Every cell below reads it from there, and `check_config()` refuses to start if the file
is not the one you meant — on 2026-08-14 a hard-coded path in STEP 5A spent 3 h training
v1 by mistake, and nothing in the epoch log revealed it.

**Fresh run (v2):** Cell 1 → 2 → 3 → 4 (smoke) → 5A (short, 20 epochs) → **STEP 8** (full).
5A is not optional boilerplate — it is where you confirm the adversarial game is healthy
before committing hours of A100 time.

**E1 history:** STEP 5B (the 400-epoch run) and STEP 6 (resume with a rebalanced
discriminator) are kept as the record of how E1 was trained. STEP 7 grades any run.

**Why this notebook instead of `SA_CUT_Colab_Train.ipynb`:** that notebook builds its
experiment YAML from a Python dict listing only four loss keys, so anything not in that
list silently falls back to `configs/default.yaml`. Since `default.yaml` keeps
`color_loss_mode: global` (so ablation baselines stay unchanged), that path would train
**without the region-conditioned colour loss**. This notebook uses a committed experiment
YAML directly and only overrides paths on the CLI, so every component stays as committed.

**Design (same three rules as the SQ-MIL bootstrap):**
1. **Code on Colab local disk**, pulled from GitHub — fast, always the committed config.
2. **Data copied Drive → local disk once per session** — training reads many small patch
   files per epoch; the Drive FUSE mount is far slower than local disk.
3. **Checkpoints written back to Drive** — Colab disconnects; weights must survive.
   The text log goes there too (`$DRIVE/results/logs/<name>/train.log`), so the epoch
   lines outlive the browser tab.

**Watch while training:** with `gan_mode: lsgan`,
`loss_D = 0.5·[(D(real)−1)² + D(fake)²]`, so `D(real) = D(fake) = 0.5` gives
**`loss_D = 0.25` — that is equilibrium, not failure.** Below ~0.15 the discriminator is
winning and the adversarial gradient to G vanishes, leaving "colourised TPAF" instead of
H&E; E1 drifted to `d_loss_ema = 0.129`, which is what STEP 6 corrected.

An earlier version of this notebook demanded 0.3–0.7. That is the vanilla/BCE scale
(equilibrium 0.693) and was never right for LSGAN. Worse, it is unreachable: setting
`d_loss_gate_threshold` above equilibrium latches the gate shut and freezes D forever.
Read `D_gate` and the real/fake split alongside `loss_D` — see STEP 5A.

Expected on a healthy run: `struct=0.0000` for the first 3 epochs (warm-up) then ramping
to `lambda_struct=5.0` over 5 epochs; `color` noticeably larger than in `global` mode.


In [ ]:
# ── Cell 1: mount Drive + config ─────────────────────────────
from google.colab import drive
import os

drive.mount('/content/drive')

# --- edit here if your paths differ ---
os.environ['DRIVE']    = '/content/drive/MyDrive/SA-CUT'
os.environ['REPO_URL'] = 'https://github.com/z-pan/SA-CUT.git'
os.environ['BRANCH']   = 'main'
os.environ['RUN_NAME']  = 'E1_sa_cut_full'          # the finished E1 run (STEP 5B/6/7)

# --- which experiment the smoke test and STEP 5A run ---
os.environ['CONFIG']     = 'configs/experiment_sa_cut_v2.yaml'
os.environ['SHORT_NAME'] = 'E1_sa_cut_v2_short'


def check_config():
    """Fail before the GPU bill, not after, if CONFIG is not the intended run.

    STEP 5A used to hard-code its config path, and on 2026-08-14 that path still
    pointed at v1 — 3 h of A100 time produced a run that could say nothing about
    the decoder under test. Nothing in the epoch log gives the config away: only
    the resolved-config dump at the very top does, and that is the first thing
    Colab's output truncates.

    Checks the three settings that define v2, so pointing CONFIG at any other
    experiment stops here with a readable message. Extend it when you add one.
    """
    import yaml
    path = os.environ['CONFIG']
    # .get(), not ['...']: experiment_sa_cut_full.yaml simply omits
    # decoder_upsample (the generator then defaults to 'transpose'), and a
    # KeyError would say far less about what went wrong than the assertion does.
    # encoding is explicit because the configs carry non-ASCII in their header
    # comments; bare open() picks the platform default and dies on a non-UTF-8
    # box (Colab is UTF-8, a Windows checkout is not).
    cfg = yaml.safe_load(open(path, encoding='utf-8'))
    assert cfg['generator'].get('decoder_upsample') == 'resize_conv', cfg['generator']
    assert cfg['training'].get('lr_D') == 5e-5, cfg['training'].get('lr_D')
    # Must stay below the LSGAN equilibrium loss of 0.25. A gate above it can
    # never reopen: 0.3 latched shut after ~77 steps and froze D for the whole
    # first v2 run (D_gate pegged at 100%, epoch time down 32%).
    assert cfg['training'].get('d_loss_gate_threshold') == 0.2, cfg['training']
    print(f"config OK: {path} -> {os.environ['SHORT_NAME']}")


assert os.path.isdir(os.environ['DRIVE']), f"Drive folder not found: {os.environ['DRIVE']}"
print('GPU:'); os.system('nvidia-smi --query-gpu=name,memory.total --format=csv,noheader')


In [ ]:
# ── Cell 2: code → local disk (clone, or pull if present) + deps ──
%cd /content
!if [ -d SA-CUT ]; then cd SA-CUT && git fetch && git checkout "$BRANCH" && git pull --ff-only; else git clone --branch "$BRANCH" "$REPO_URL"; fi
%cd /content/SA-CUT
!pip install -q tifffile pytorch-fid pyyaml scikit-image
# Confirm the region-conditioned colour loss commit is present.
!git log --oneline -3
!test -f losses/region_color_loss.py && echo 'OK: region_color_loss.py present'

In [ ]:
# ── Cell 3: data Drive → local disk (once per session) ──
# rsync --ignore-existing makes reconnects near-instant.
!mkdir -p data/raw/tpaf data/raw/he data/patches/masks
!rsync -a --ignore-existing "$DRIVE/patches/tpaf/"  data/raw/tpaf/
!rsync -a --ignore-existing "$DRIVE/patches/he/"    data/raw/he/
!rsync -a --ignore-existing "$DRIVE/patches/masks/" data/patches/masks/

# Masks are matched to TPAF patches by filename stem, so a mask must exist for
# every TPAF patch in precomputed mode.
!echo "tpaf=$(ls data/raw/tpaf | wc -l)  he=$(ls data/raw/he | wc -l)  masks=$(ls data/patches/masks | wc -l)"

In [ ]:
# ── Cell 4: smoke test first (~30 s) ──
# Same check as locally: CPU, 1 epoch, tiny synthetic data. Verifies the Colab
# environment and the committed code before spending A100 hours — on the same
# config STEP 5A will train, so a broken experiment YAML surfaces here.
check_config()
!bash scripts/smoke_test.sh --config "$CONFIG"


## STEP 5A — short validation run (RUN THIS FIRST)

**Do not skip ahead to the full run.** 20 epochs at constant LR, ~1/20th of the cost, to
confirm the run is healthy before committing hours of A100 time.

This cell now pre-flights **SA-CUT v2** (`experiment_sa_cut_v2.yaml`, the resize_conv
decoder under test in STEP 8). E1 itself finished at epoch 399, so STEP 5B and STEP 6
below are kept as the record of how it was trained, not as work still to do.

Check in the epoch log:

| Field | Healthy | Bad |
|---|---|---|
| `D=` | near **0.25** | `< 0.15` → D is winning; G degrades to colourised TPAF |
| `D= (real … / fake …)` | the two stay clearly apart | both drifting to ~0.25 → D is uninformative |
| `D_gate=` | low % | pegged near 100% → D is frozen and nothing is being tested |
| `struct=` | `0.0000` for epochs 0–2, then rising | still 0 after epoch 5 |
| `color=` | clearly non-trivial (region mode) | ~0 |
| `G=` | fluctuating, no NaN / blow-up | NaN or monotonic explosion |

**Why 0.25 and not the 0.3–0.7 this table used to say.** With `gan_mode: lsgan`,
`loss_D = 0.5·[(D(real)−1)² + D(fake)²]`, so the balance point `D(real) = D(fake) = 0.5`
gives **0.25**. The 0.3–0.7 band is the vanilla/BCE scale (equilibrium 0.693) and never
applied here.

`loss_D` alone cannot tell a balanced discriminator from a dead one — a D emitting ~0.5
for everything also scores 0.25. That is why the real/fake split and `D_gate` are in the
table: the first v2 run sat at a healthy-looking 0.26 while `D_gate` was pegged at 100%
and D had taken ~77 gradient steps in 51k iterations.

Also open a sample image under `$DRIVE/results/logs/` — nuclei should be trending purple,
not blank/white (blank nuclei is the UTOM failure mode SA-CUT exists to fix).

Only if all of the above look right, continue to the full run (STEP 8 for v2).


In [ ]:
# ── STEP 5A: short validation run — 20 epochs, no LR decay ──
# Config and name come from Cell 1; separate name so this never overwrites the
# real E1 checkpoints/logs.
check_config()

!python scripts/train.py \
    --config "$CONFIG" \
    --training.n_epochs=20 \
    --training.n_epochs_decay=0 \
    --data.patch_size=512 \
    --data.num_workers=2 \
    --experiment.name="$SHORT_NAME" \
    --experiment.checkpoint_dir="$DRIVE/checkpoints" \
    --experiment.log_dir="$DRIVE/results/logs" \
    --experiment.use_wandb=false

# The text log is written to $DRIVE/results/logs/$SHORT_NAME/train.log, so the
# epoch lines survive a disconnect even if this cell's output does not.


## STEP 5B — full E1 run (only after 5A looks healthy)

400 epochs (200 at full LR + 200 linear decay) — the reference run for the Ch5 experiment
matrix. Expect at least one Colab disconnect; checkpoints go to Drive every 10 epochs, so
use the resume cell below.

The first 200 epochs run at constant LR, so **STEP 5A's epochs are numerically identical to
the opening of this run** (same seed, same LR). 5A therefore costs nothing in information
terms — it just lets you bail out early instead of hours in.

In [ ]:
# ── STEP 5B: full E1 training (400 epochs) ──
# Config is used as committed (mask input + SA-PatchNCE + L_struct + region colour
# loss + the D-collapse guards). Only paths and patch size are overridden.
# patch_size=512 matches the 512x512 precomputed masks on Drive.
!python scripts/train.py \
    --config configs/experiment_sa_cut_full.yaml \
    --data.patch_size=512 \
    --data.num_workers=2 \
    --experiment.name="$RUN_NAME" \
    --experiment.checkpoint_dir="$DRIVE/checkpoints" \
    --experiment.log_dir="$DRIVE/results/logs" \
    --experiment.use_wandb=false

## STEP 6 — resume with a rebalanced discriminator

**Read this before resuming.** The first E1 run stopped at **epoch 186/400** with
`d_loss_ema = 0.129` — well below the mandatory 0.3–0.7 band. D had won, so G was getting
a weak/uninformative adversarial signal. Measured on 40 tissue-rich test patches, the
epoch-186 generator gives:

| | epoch 186 | UTOM nuc_hi | real H&E |
|---|---|---|---|
| cytoplasm R−B gap | **−0.1** (solved) | −10.9 (too violet) | 0 |
| nuclei R−B gap | +16.4 (too weak) | +8.4 | 0 |
| **green-dominant pixels** | **13.61 %** | 0.51 % | 0.01 % |

So the region-conditioned colour loss did fix the violet cytoplasm, but the generator is
also painting colours that cannot exist in H&E (13.6 % green) and under-staining nuclei —
both consistent with an over-strong D that G is gaming rather than matching.

**What changed below, and why:**

* `d_loss_gate_threshold` 0.1 → **0.3** — the direct lever. D updates are skipped while
  `EMA(loss_D) < 0.3`, so D is held back until G catches up, then gating disengages by
  itself. Self-regulating, which is why it is the primary change.
* `lr_D` 1e-4 → **5e-5** — makes D adapt more slowly from here on, so it does not simply
  re-win between gated steps.

Everything else is left exactly as trained. Watch `D=` climb into 0.3–0.7 over the first
few epochs; `D_gate=` should start high and fall as G recovers. If green pixels are still
above ~2 % after ~20 more epochs, the next lever is an explicit colour constraint rather
than more D tuning.

In [ ]:
# -- STEP 6: resume E1 from the last checkpoint, with D held back --
# Re-run Cells 1-3 first after a disconnect (Cell 3 is fast on reconnect).
# The two overrides fix the over-strong discriminator; everything else is as trained.
!python scripts/train.py \
    --config configs/experiment_sa_cut_full.yaml \
    --resume "$DRIVE/checkpoints/$RUN_NAME/latest.pth" \
    --training.d_loss_gate_threshold=0.3 \
    --training.lr_D=5e-5 \
    --data.patch_size=512 \
    --data.num_workers=2 \
    --experiment.name="$RUN_NAME" \
    --experiment.checkpoint_dir="$DRIVE/checkpoints" \
    --experiment.log_dir="$DRIVE/results/logs" \
    --experiment.use_wandb=false

## STEP 7 — grade the checkpoint objectively

Run this after (or during) training to decide whether the rebalance worked, instead of
judging by eye. It translates a fixed set of TPAF patches and scores the result against
real H&E.

Targets to beat, from the epoch-186 baseline:

| metric | epoch 186 | goal |
|---|---|---|
| cytoplasm R−B gap | −0.1 | keep near 0 |
| nuclei R−B gap | +16.4 | → 0 (more violet); UTOM reaches +8.4 |
| green-dominant pixels | 13.61 % | **< 1 %** |

The green figure is the one that matters most right now — a run can score a perfect
cytoplasm hue while painting impossible colours, which is exactly what epoch 186 did.

**If the log prints an unexpected epoch**, the checkpoint path is wrong. The trainer
appends `experiment.name` to `checkpoint_dir`, so weights end up in
`$DRIVE/checkpoints/$RUN_NAME/`. Passing `checkpoints/$RUN_NAME` as `checkpoint_dir`
nests them one level deeper and the old file is read instead — check the epoch in the
first log line before trusting the scores.

In [ ]:
# -- STEP 7: translate a held-out set, then score it --
# The trainer appends experiment.name to checkpoint_dir, so the weights live in
# $DRIVE/checkpoints/$RUN_NAME/ -- pass the parent, not the run folder, above.
!python scripts/test.py     --checkpoint "$DRIVE/checkpoints/$RUN_NAME/latest.pth"     --input_dir  data/raw/tpaf     --mask_dir   data/patches/masks     --output_dir results/eval_$RUN_NAME

# Real H&E is the reference domain; masks define nuclei on the generated side.
!python scripts/eval_stain_color.py     --virtual  results/eval_$RUN_NAME     --nuc-mask data/patches/masks     --real     data/raw/he

## STEP 8 — v2: checkerboard-free decoder (train from scratch)

E1 finished at epoch 399 with `d_loss_ema = 0.173` — still under the 0.3–0.7 band —
and, measured on a fixed 40-patch set:

| | epoch 186 | epoch 399 | real |
|---|---|---|---|
| cytoplasm R−B gap | −0.1 | +5.6 | 0 |
| nuclei R−B gap | +16.4 | +15.7 | 0 |
| green-dominant px | 13.61 % | 9.34 % | 0.01 % |

Rebalancing D moved every number the right way but not far enough, and visually it
got *worse*: the large green blotches became a fine periodic weave. Both are the same
mechanism — the decoder upsamples with `ConvTranspose2d(kernel=3, stride=2)`, and 3 is
not divisible by 2, so output positions receive uneven numbers of kernel weights and
each stage doubles the imbalance. D never recovered because those artifacts make every
generated patch trivially identifiable.

`configs/experiment_sa_cut_v2.yaml` swaps that decoder for nearest-neighbour upsample +
stride-1 conv, and keeps the D settings from the resume. **This changes weight shapes,
so it trains from scratch — it cannot resume from an E1 checkpoint.**

Run STEP 5A first with this config (`--config configs/experiment_sa_cut_v2.yaml`) to
confirm `loss_D` behaves before committing the full run. Success criterion: green-dominant
pixels below ~1 % in STEP 7. If they persist, the next levers are `norm_type`
(instance → group) and an explicit colour-gamut penalty.

In [ ]:
# -- STEP 8: SA-CUT v2, from scratch --
!python scripts/train.py     --config configs/experiment_sa_cut_v2.yaml     --data.patch_size=512     --data.num_workers=2     --experiment.name=sa_cut_v2     --experiment.checkpoint_dir="$DRIVE/checkpoints"     --experiment.log_dir="$DRIVE/results/logs"     --experiment.use_wandb=false

# Then grade it the same way (STEP 7), pointing at the v2 run:
# !python scripts/test.py --checkpoint "$DRIVE/checkpoints/sa_cut_v2/latest.pth" #     --input_dir data/raw/tpaf --mask_dir data/patches/masks #     --output_dir results/eval_sa_cut_v2
# !python scripts/eval_stain_color.py --virtual results/eval_sa_cut_v2 #     --nuc-mask data/patches/masks --real data/raw/he

## Next: E2 baselines and E3 ablations

Same pattern — swap the config, keep the path overrides and `RUN_NAME`. The ablation
configs already exist in `configs/`, so no dict-built YAML is needed:

| Experiment | Config |
|---|---|
| E2 CUT baseline | `configs/ablation_cut_baseline.yaml` |
| E2 CycleGAN | `configs/ablation_cyclegan.yaml` |
| E3 mask input only | `configs/ablation_cut_mask_input.yaml` |
| E3 no `L_struct` | `configs/ablation_sa_cut_no_struct.yaml` |

Give each run its own `RUN_NAME` so checkpoints and logs stay separate on Drive.